<a href="https://colab.research.google.com/github/Ekrem-Guler/RAG-Based-Mathematical-Problem-Solver/blob/main/RAG_MATH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import chromadb

In [ ]:
from datasets import load_dataset

ds = load_dataset("EleutherAI/hendrycks_math", "counting_and_probability")
ds


In [ ]:
ds_geo = load_dataset("EleutherAI/hendrycks_math", "geometry")
ds_geo

In [ ]:
ds_alg = load_dataset("EleutherAI/hendrycks_math", "algebra")
ds_alg

In [ ]:
sentences = [ds["train"][i]["solution"] for i in range(len(ds["train"]))]

In [ ]:
sentences_test = [ds["test"][i]["solution"] for i in range(len(ds["test"]))]

In [ ]:
sentences_geo = [ds_geo["train"][i]["solution"] for i in range(len(ds_geo["train"]))]

In [ ]:
sentences_geo_test = [ds_geo["test"][i]["solution"] for i in range(len(ds_geo["test"]))]

In [ ]:
sentences_alg = [ds_alg["train"][i]["solution"] for i in range(len(ds_alg["train"]))]

In [ ]:
sentences_alg_test = [ds_alg["test"][i]["solution"] for i in range(len(ds_alg["test"]))]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(sentences)
print(len(embeddings[0]))


In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings_geo = model.encode(sentences_geo)
print(len(embeddings_geo[0]))

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings_alg = model.encode(sentences_alg)
print(len(embeddings_alg[0]))

In [ ]:

chroma_client = chromadb.Client()
collection_math = chroma_client.create_collection(name="math_collection",configuration={"hnsw": {"space": "cosine"}})


In [ ]:
collection_math.add(
    documents=sentences,
    ids = [str(i) for i in range(771)],
    embeddings=embeddings.tolist()
)


In [ ]:
collection_math.add(
    documents=sentences_geo,
    ids = [str(i+771) for i in range(870)],
    embeddings=embeddings_geo.tolist()
)


In [ ]:

collection_math.add(
    documents=sentences_alg,
    ids = [str(i+771+870) for i in range(1744)],
    embeddings=embeddings_alg.tolist()
)


In [ ]:
results = collection_math.query(
    query_texts=[
        "How many natural numbers greater than 6 but less than 60 are relatively prime to 15?"
    ],
    n_results=10,
    include=["embeddings", "documents", "distances"]
)

retrieved_solution = results['documents'][0][0]
print(f"Retrieved Solution: {retrieved_solution}")
for i in range(len(results['ids'][0])):
    print(f"ID: {results['ids'][0][i]}")
    print(f"Distance: {results['distances'][0][i]}") # Lower is better!
    print(f"Content snippet: {results['documents'][0][i][:50]}...")